# Kaggle training — single_drug_toxicity

Run this notebook only after Colab preparation and audit are complete. Pull the same GitHub revision and point AUDITDDI_DATA_BASE to the Kaggle-visible linked Drive folder.


In [ ]:
from pathlib import Path
import os, sys, subprocess

# Kaggle is reserved for model fitting. Use the GitHub checkout for code and the linked Drive mount for data.
DATA_VALUE = os.environ.get("AUDITDDI_DATA_BASE", "").strip()
assert DATA_VALUE, "Set AUDITDDI_DATA_BASE to the Kaggle-visible path for MyDrive/Projects/auditddi."
DATA = Path(DATA_VALUE).expanduser().resolve()
assert DATA.is_dir(), f"Linked Drive data root not found: {DATA}"
REPO_VALUE = os.environ.get("AUDITDDI_CODE_ROOT", "").strip()
REPO_CANDIDATES = [Path(REPO_VALUE)] if REPO_VALUE else []
REPO_CANDIDATES += [Path.cwd(), Path("/kaggle/working/auditddi"), Path("/kaggle/working/auditddi_v2")]
REPO_CANDIDATES += list(Path.cwd().parents)
REPO = next((p.resolve() for p in REPO_CANDIDATES if (p / "src").is_dir() and (p / "requirements_kaggle.txt").is_file()), None)
if REPO is None:
    raise FileNotFoundError("GitHub checkout not found. Clone/pull https://github.com/akhilau-git/AuditDDI-project.git into Kaggle and set AUDITDDI_CODE_ROOT if needed.")
RESULTS_BASE = Path(os.environ.get("AUDITDDI_RESULTS_BASE", str(DATA / "results"))).expanduser()
os.environ["AUDITDDI_DATA_BASE"] = str(DATA)
os.environ["AUDITDDI_RESULTS_BASE"] = str(RESULTS_BASE)
sys.path.insert(0, str(REPO / "src"))
RUNS = RESULTS_BASE / "pillars" / "single_drug_toxicity"
RUNS.mkdir(parents=True, exist_ok=True)
print("Kaggle training environment; code:", REPO)
print("Linked Drive data:", DATA)
print("Pillar results:", RUNS)


In [ ]:
%pip install -r {REPO / "requirements_kaggle.txt"}


In [ ]:
# Kaggle model fit. This outcome is a FAERS report fraction, not clinical toxicity incidence.
LABELS = DATA / "processed/seven_branch/single_drug_toxicity_labels_candidate_2026q2.csv"
assert LABELS.is_file(), f"Prepared labels not found: {LABELS}"
OUT = RUNS / "beta_binomial_2026q2"
OUT.mkdir(parents=True, exist_ok=True)
subprocess.run([sys.executable, "-u", str(REPO / "pillars/single_drug_toxicity/scripts/run_beta_binomial.py"), "--labels", str(LABELS), "--output-dir", str(OUT), "--seeds", "203", "204", "205", "206", "207"], cwd=REPO, check=True)
